# OpenMHC Quickstart

End-to-end walkthrough of the public evaluation API:

1. Download the tiny dataset
2. Track 2 — define a trivial `Imputer` and run `evaluate_imputation`
3. Track 1 — define a trivial `Encoder` and run `evaluate_prediction`
4. Track 3 — define a trivial `Forecaster` and run `evaluate_forecasting`
5. Generate a paste-ready submission body

This notebook is the smallest end-to-end loop. It runs on CPU and finishes in a few minutes against the tiny version.

## 1. Install + download the dataset

If you haven't already:

```bash
pip install -e ..
```

Then download the XS dataset (~TBD MB).

In [1]:
import os
from pathlib import Path

import openmhc

# Pick a destination, or set MHC_DATA_DIR before running this cell.
DATA_ROOT = Path(os.environ.get("MHC_DATA_DIR", "~/.cache/openmhc/data-xs")).expanduser()

# Download only if not already present; otherwise reuse the cached copy.
if not (DATA_ROOT / "dataset_version.json").exists():
    openmhc.download_dataset(version="xs", dest=DATA_ROOT)

# Make this the dataset root for every subsequent openmhc.* call.
os.environ["MHC_DATA_DIR"] = str(DATA_ROOT)
print(f"dataset is at: {DATA_ROOT}")

dataset is at: /home/narayanschuetz/.cache/openmhc/data-xs


## 2. Define a trivial Imputer

Models implement the `Imputer` protocol — one method, no inheritance:
- `impute(data, observed_mask, target_mask)` — fill in artificially-masked positions

All setup (loading checkpoints, computing statistics from the training split, building per-user state) happens in `__init__`; the benchmark harness does not call any preparation method. Below we compute the per-channel global mean once from the train split via `openmhc.iter_train_data(version=VERSION)`, then use it as the fill value.

`VERSION` is set once and threaded through every public-API call. There is no auto-detection: the dataset root carries a `dataset_version.json` marker and the resolver insists the version you pass matches it.

In [2]:
import numpy as np
import openmhc

# Match what was downloaded above. Switch to "full" once you have the full release.
VERSION = "xs"


class MeanImputer:
    """Predict the channel-wise mean for every masked position."""

    def __init__(self):
        # Compute per-channel mean from the official train split.
        data_sum = np.zeros(19, dtype=np.float64)
        data_count = np.zeros(19, dtype=np.float64)
        for data, mask in openmhc.iter_train_data(version=VERSION):
            obs = (mask > 0.5) & np.isfinite(data)
            data_sum += np.where(obs, data, 0.0).sum(axis=(0, 2))
            data_count += obs.sum(axis=(0, 2))
        self.means = (data_sum / np.maximum(data_count, 1)).astype(np.float32)

    def impute(
        self,
        data: np.ndarray,
        observed_mask: np.ndarray,
        target_mask: np.ndarray,
    ) -> np.ndarray:
        out = data.copy()
        for ch in range(19):
            target = target_mask[:, ch, :] == 1
            out[:, ch, :][target] = self.means[ch]
        return out.astype(np.float32)

## 3. Run imputation evaluation

`evaluate_imputation` runs your imputer against all 6 masking scenarios (random noise, temporal slices, sleep gaps, etc.) on the test split.

In [3]:
results = openmhc.evaluate_imputation(MeanImputer(), version=VERSION)
results.summary()

Loading dataset from disk:   0%|          | 0/31 [00:00<?, ?it/s]

Filter (num_proc=1): 100%|##########| 136978/136978 [00:00<?, ? examples/s]

Filter (num_proc=1): 100%|##########| 56587/56587 [00:00<?, ? examples/s]

Loading dataset from disk:   0%|          | 0/31 [00:00<?, ?it/s]

Filter (num_proc=1): 100%|##########| 136978/136978 [00:00<?, ? examples/s]

Filter (num_proc=1): 100%|##########| 56587/56587 [00:00<?, ? examples/s]

metric,scenario,split,channel_group,macro_balanced_accuracy,macro_roc_auc,mean_normalized_mae,mean_normalized_mse,mean_normalized_rmse,n_channels
0,intensity_failure,test,binary,NaN,NaN,NaN,NaN,NaN,0
1,intensity_failure,test,continuous,NaN,NaN,6.279649,48.10646,6.821735,2
2,intensity_failure,val,binary,NaN,NaN,NaN,NaN,NaN,0
3,intensity_failure,val,continuous,NaN,NaN,5.48789,34.868949,5.895446,2
4,random_noise,test,binary,0.5,0.5,NaN,NaN,NaN,12
5,random_noise,test,continuous,NaN,NaN,0.475102,1.070293,1.030996,7
6,random_noise,val,binary,0.5,0.5,NaN,NaN,NaN,9
7,random_noise,val,continuous,NaN,NaN,0.454351,0.856238,0.914558,7
8,signal_slice,test,binary,0.5,0.5,NaN,NaN,NaN,12
9,signal_slice,test,continuous,NaN,NaN,0.476324,1.072474,1.03175,7


## 4. Define a trivial Encoder

For Track 1 (outcome prediction) the protocol is just `encode(weekly_tensors) -> embeddings`. Here we mean-pool over the time axis.

In [4]:
class MeanPoolEncoder:
    """Average over the 168 hours, take the 19 sensor channels."""

    def encode(self, weekly_tensors: np.ndarray) -> np.ndarray:
        # weekly_tensors: (B, 168, 38). Cols 0-18 sensors, 19-37 missingness.
        return weekly_tensors[:, :, :19].mean(axis=1).astype(np.float32)

## 5. Run prediction evaluation (Track 1)

`evaluate_prediction` extracts your embeddings, then fits a linear probe per task (logistic regression for binary, ordinal logistic for ordinal, ridge regression for continuous).

In [5]:
# tasks="all" is fine on the XS dataset; restrict if you want a faster loop
results = openmhc.evaluate_prediction(
    MeanPoolEncoder(), version=VERSION, tasks=["BiologicalSex", "age"]
)
results.summary()

Encoding:   0%|          | 0/112 [00:00<?, ?batch/s]

Encoding:   1%|          | 1/112 [00:00<00:34,  3.25batch/s]

Encoding:   2%|▏         | 2/112 [00:00<00:32,  3.40batch/s]

Encoding:   3%|▎         | 3/112 [00:00<00:31,  3.42batch/s]

Encoding:   4%|▎         | 4/112 [00:01<00:30,  3.50batch/s]

...

Encoding:  98%|█████████▊| 110/112 [00:33<00:00,  3.21batch/s]

Encoding:  99%|█████████▉| 111/112 [00:34<00:00,  3.24batch/s]

Encoding: 100%|██████████| 112/112 [00:34<00:00,  3.28batch/s]

/home/narayanschuetz/myheartcounts-dataset/src/openmhc/_evaluate.py:483: UserWarning: Predictions are constant (zero variance). Pearson correlation will be NaN.
  m = compute_regression_metrics(y_test, test_pred)


metric,task,task_type,auprc,auroc,mae,mse,r2
0,BiologicalSex,binary,0.694444,0.5,NaN,NaN,NaN
1,age,regression,NaN,NaN,11.872222,202.545556,-0.001557


## 7. Generate a leaderboard submission

`results.to_submission_yaml(...)` produces a paste-ready body matching the textareas in the [submission issue template](https://github.com/AshleyLab/myheartcounts-dataset/issues/new?template=submission.yml).

For Track 2 imputation, skill scores and per-category subgroup scores are computed locally against the frozen LOCF baseline (`data/baselines/imputation_locf.json`). For Track 1 and Track 3, those fields are emitted as `—` and filled in by the maintainers from `raw_metrics` during ingestion.

In [6]:
class LastValueForecaster:
    """Repeat the most recent observed value across the forecast horizon."""

    def predict(self, history: np.ndarray, horizon: int) -> np.ndarray:
        # history: (n_channels, history_length); returns (n_channels, horizon)
        last = np.nan_to_num(history[:, -1:], nan=0.0)
        return np.tile(last, (1, horizon)).astype(np.float32)


# max_samples keeps this fast on the tiny dataset; remove for full eval
forecast_results = openmhc.evaluate_forecasting(
    LastValueForecaster(), version=VERSION, forecasting_length=24, max_samples=20
)
forecast_results.summary()


████████╗██╗███╗   ███╗███████╗    ███████╗███████╗██████╗ ██╗███████╗███████╗    █████╗ ██╗
╚══██╔══╝██║████╗ ████║██╔════╝    ██╔════╝██╔════╝██╔══██╗██║██╔════╝██╔════╝   ██╔══██╗██║
   ██║   ██║██╔████╔██║█████╗█████╗███████╗█████╗  ██████╔╝██║█████╗  ███████╗   ███████║██║
   ██║   ██║██║╚██╔╝██║██╔══╝╚════╝╚════██║██╔══╝  ██╔══██╗██║██╔══╝  ╚════██║   ██╔══██║██║
   ██║   ██║██║ ╚═╝ ██║███████╗    ███████║███████╗██║  ██║██║███████╗███████║██╗██║  ██║██║
   ╚═╝   ╚═╝╚═╝     ╚═╝╚══════╝    ╚══════╝╚══════╝╚═╝  ╚═╝╚═╝╚══════╝╚══════╝╚═╝╚═╝  ╚═╝╚═╝
ai4ts v0.0.3 - building AI for unified time-series analysis, https://time-series.ai 




                      FORECASTING EVALUATION CONFIGURATION                      


┌─ Seed                 42


│


├─ Experiment Name      "openmhc_run"


...

/home/narayanschuetz/myheartcounts-dataset/src/forecasting_evaluation/runner.py:160: RuntimeWarning: Mean of empty slice
  ch_mean = float(np.nanmean(ch_arr)) if ch_arr.size else float("nan")
/home/narayanschuetz/myheartcounts-dataset/src/forecasting_evaluation/runner.py:160: RuntimeWarning: Mean of empty slice
  ch_mean = float(np.nanmean(ch_arr)) if ch_arr.size else float("nan")
/home/narayanschuetz/myheartcounts-dataset/src/forecasting_evaluation/runner.py:160: RuntimeWarning: Mean of empty slice
  ch_mean = float(np.nanmean(ch_arr)) if ch_arr.size else float("nan")
/home/narayanschuetz/myheartcounts-dataset/src/forecasting_evaluation/runner.py:160: RuntimeWarning: Mean of empty slice
  ch_mean = float(np.nanmean(ch_arr)) if ch_arr.size else float("nan")
/home/narayanschuetz/myheartcounts-dataset/src/forecasting_evaluation/runner.py:160: RuntimeWarning: Mean of empty slice
  ch_mean = float(np.nanmean(ch_arr)) if ch_arr.size else float("nan")
/home/narayanschuetz/myheartcounts-datas

/home/narayanschuetz/myheartcounts-dataset/src/forecasting_evaluation/runner.py:160: RuntimeWarning: Mean of empty slice
  ch_mean = float(np.nanmean(ch_arr)) if ch_arr.size else float("nan")
/home/narayanschuetz/myheartcounts-dataset/src/forecasting_evaluation/runner.py:160: RuntimeWarning: Mean of empty slice
  ch_mean = float(np.nanmean(ch_arr)) if ch_arr.size else float("nan")
/home/narayanschuetz/myheartcounts-dataset/src/forecasting_evaluation/runner.py:160: RuntimeWarning: Mean of empty slice
  ch_mean = float(np.nanmean(ch_arr)) if ch_arr.size else float("nan")
/home/narayanschuetz/myheartcounts-dataset/src/forecasting_evaluation/runner.py:160: RuntimeWarning: Mean of empty slice
  ch_mean = float(np.nanmean(ch_arr)) if ch_arr.size else float("nan")
/home/narayanschuetz/myheartcounts-dataset/src/forecasting_evaluation/runner.py:160: RuntimeWarning: Mean of empty slice
  ch_mean = float(np.nanmean(ch_arr)) if ch_arr.size else float("nan")
/home/narayanschuetz/myheartcounts-datas

/home/narayanschuetz/myheartcounts-dataset/src/forecasting_evaluation/runner.py:160: RuntimeWarning: Mean of empty slice
  ch_mean = float(np.nanmean(ch_arr)) if ch_arr.size else float("nan")
/home/narayanschuetz/myheartcounts-dataset/src/forecasting_evaluation/runner.py:160: RuntimeWarning: Mean of empty slice
  ch_mean = float(np.nanmean(ch_arr)) if ch_arr.size else float("nan")
/home/narayanschuetz/myheartcounts-dataset/src/forecasting_evaluation/runner.py:160: RuntimeWarning: Mean of empty slice
  ch_mean = float(np.nanmean(ch_arr)) if ch_arr.size else float("nan")
/home/narayanschuetz/myheartcounts-dataset/src/forecasting_evaluation/runner.py:160: RuntimeWarning: Mean of empty slice
  ch_mean = float(np.nanmean(ch_arr)) if ch_arr.size else float("nan")
/home/narayanschuetz/myheartcounts-dataset/src/forecasting_evaluation/runner.py:160: RuntimeWarning: Mean of empty slice
  ch_mean = float(np.nanmean(ch_arr)) if ch_arr.size else float("nan")
/home/narayanschuetz/myheartcounts-datas

/home/narayanschuetz/myheartcounts-dataset/src/forecasting_evaluation/runner.py:160: RuntimeWarning: Mean of empty slice
  ch_mean = float(np.nanmean(ch_arr)) if ch_arr.size else float("nan")
/home/narayanschuetz/myheartcounts-dataset/src/forecasting_evaluation/runner.py:160: RuntimeWarning: Mean of empty slice
  ch_mean = float(np.nanmean(ch_arr)) if ch_arr.size else float("nan")
/home/narayanschuetz/myheartcounts-dataset/src/forecasting_evaluation/runner.py:160: RuntimeWarning: Mean of empty slice
  ch_mean = float(np.nanmean(ch_arr)) if ch_arr.size else float("nan")
/home/narayanschuetz/myheartcounts-dataset/src/forecasting_evaluation/runner.py:160: RuntimeWarning: Mean of empty slice
  ch_mean = float(np.nanmean(ch_arr)) if ch_arr.size else float("nan")


metric,channel,mae,mase,mase_all,mse,ql,sql
0,ch_0,172.406827,1.126707,0.843561,212979.762049,86.203413,0.563354
1,ch_1,138.726720,1.409292,0.963318,149358.500563,69.363360,0.704646
2,ch_2,0.000000,NaN,NaN,0.000000,0.000000,NaN
3,ch_3,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,ch_4,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
5,ch_5,0.145562,1.858889,1.685999,0.045959,0.072781,0.929444
6,ch_6,239.186330,1.082481,0.884697,266894.592831,119.593165,0.541240
7,ch_9,0.068182,0.727273,1.636364,0.068182,0.034091,0.363636


## 6. Generate a leaderboard submission

`results.to_submission_yaml(...)` produces a paste-ready body matching the textareas in the [submission issue template](https://github.com/AshleyLab/myheartcounts-dataset/issues/new?template=submission.yml).

For Track 2 imputation, skill scores and per-category subgroup scores are computed locally against the frozen LOCF baseline (`data/baselines/imputation_locf.json`). For Track 1, those fields are emitted as `—` and filled in by the maintainers from `raw_metrics` during ingestion.

In [7]:
imputation_results = openmhc.evaluate_imputation(MeanImputer(), version=VERSION)

submission_body = imputation_results.to_submission_yaml(
    method_name="MeanImputer (demo)",
    submitter_team="Stanford CS",
    paper_url="https://arxiv.org/abs/0000.00000",
    code_url="https://github.com/example/mean-imputer",
    method_category="Statistical / Classical baseline",
)
print(submission_body)

Loading dataset from disk:   0%|          | 0/31 [00:00<?, ?it/s]

Filter (num_proc=1): 100%|##########| 136978/136978 [00:00<?, ? examples/s]

Filter (num_proc=1): 100%|##########| 56587/56587 [00:00<?, ? examples/s]

Loading dataset from disk:   0%|          | 0/31 [00:00<?, ?it/s]

Filter (num_proc=1): 100%|##########| 136978/136978 [00:00<?, ? examples/s]

Filter (num_proc=1): 100%|##########| 56587/56587 [00:00<?, ? examples/s]

# Paste-ready submission body for: MeanImputer (demo)
# Track: Track 2 — Imputation (Daily, single-day context)
#
# Single-input fields (top of the form):
#   method_name:       MeanImputer (demo)
#   submitter_team:    Stanford CS
#   track:             Track 2 — Imputation (Daily, single-day context)
#   method_category:   Statistical / Classical baseline
#   foundation_variant: N/A (not a foundation model)
#   feature_dim:       —
#   paper_url:         https://arxiv.org/abs/0000.00000
#   code_url:          https://github.com/example/mean-imputer

# --- aggregate_metrics (textarea) ---
skill_score: -31.60
fair_skill_score: —  # computed by maintainers from subgroup runs
avg_rank: —  # computed by maintainers vs current leaderboard

# --- subgroup_metrics (textarea) ---
activity: 10.23
physiology: -5.87
sleep: -140.04
workouts: -48.09
semantic: —  # paper-only category; ignore unless reporting it explicitly

# --- raw_metrics (optional textarea) ---
{
  "random_noise": {
    "val": 

Once you have a body that looks right:

1. Open a [submission issue](https://github.com/AshleyLab/myheartcounts-dataset/issues/new?template=submission.yml).
2. Fill in the single-input fields (method name, paper URL, …) and paste the three textarea blocks above.
3. The HuggingFace Space ingests merged submissions and the public leaderboard at `https://myheartcounts.stanford.edu/benchmark` rebuilds automatically.

Submissions must follow the standard protocol — same split file, masking config, label-validity criterion as the paper. The submission template enforces required fields.